# 01 — Market Regime Dashboard: SPY, QQQ and VIX

## Why this project exists

A trader often starts with a loose question: **"Are we in a risk-on or risk-off environment?"**

My goal in this notebook is not to predict tomorrow's market. The goal is to build a practical dashboard that gives a trader fast context:

- Are equities trending above or below key moving averages?
- Is volatility rising or falling?
- Are drawdowns becoming material?
- Does the current regime look calm, fragile, or stressed?

This is the kind of support work I enjoy: translating a market question into clean data, clear metrics, and an intuitive visual output.

In [ ]:
!pip install yfinance plotly -q

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import plotly.express as px
import plotly.graph_objects as go

pd.options.display.float_format = "{:,.4f}".format

## 1. Download market data

I use Yahoo Finance through `yfinance` because it is fast enough for prototyping and public portfolio work. For a professional trading desk, I would connect the same logic to a licensed market data source or internal database.

In [ ]:
tickers = ["SPY", "QQQ", "IWM", "TLT", "GLD", "UUP", "^VIX"]
start_date = "2018-01-01"

data = yf.download(tickers, start=start_date, auto_adjust=True, progress=False)
prices = data["Close"].dropna(how="all")
prices.tail()

## 2. Engineer regime features

I want features that are simple enough for a non-technical trader to interpret:

- **1-day, 5-day and 21-day returns**: short-term price pressure
- **50-day and 200-day moving averages**: trend context
- **21-day annualized volatility**: recent realized risk
- **drawdown**: distance from previous high

The point is not complexity. The point is to build metrics that can be trusted and discussed quickly.

In [ ]:
spy = prices["SPY"].copy()
features = pd.DataFrame(index=spy.index)
features["spy_close"] = spy
features["return_1d"] = spy.pct_change()
features["return_5d"] = spy.pct_change(5)
features["return_21d"] = spy.pct_change(21)
features["ma_50"] = spy.rolling(50).mean()
features["ma_200"] = spy.rolling(200).mean()
features["vol_21d_ann"] = features["return_1d"].rolling(21).std() * np.sqrt(252)
features["drawdown"] = spy / spy.cummax() - 1
features["vix"] = prices["^VIX"]
features.tail()

## 3. Define a simple market regime label

This is deliberately practical and transparent:

- **Risk-on**: SPY above 50-day and 200-day moving averages, VIX below 20
- **Risk-off**: SPY below 200-day moving average or VIX above 30
- **Neutral**: everything in between

A trader could challenge these thresholds, and that is exactly the point: the notebook makes the assumptions visible and easy to adjust.

In [ ]:
def classify_regime(row):
    if row["spy_close"] > row["ma_50"] and row["spy_close"] > row["ma_200"] and row["vix"] < 20:
        return "Risk-on"
    if row["spy_close"] < row["ma_200"] or row["vix"] > 30:
        return "Risk-off"
    return "Neutral"

features["regime"] = features.apply(classify_regime, axis=1)
features[["spy_close", "ma_50", "ma_200", "vix", "vol_21d_ann", "drawdown", "regime"]].dropna().tail(10)

## 4. Visualize the dashboard metrics

For a support analyst role, I want the output to be easy to read. A dashboard is only useful if the trader can understand it within seconds.

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=features.index, y=features["spy_close"], name="SPY"))
fig.add_trace(go.Scatter(x=features.index, y=features["ma_50"], name="50D MA"))
fig.add_trace(go.Scatter(x=features.index, y=features["ma_200"], name="200D MA"))
fig.update_layout(title="SPY trend dashboard", height=500)
fig.show()

px.line(features, y=["vol_21d_ann", "drawdown", "vix"], title="Risk dashboard: realized volatility, drawdown and VIX").show()

## 5. My conclusion

This dashboard gives a compact view of market context. It does not tell a trader what to do. It helps frame the conversation:

- If SPY is above trend and volatility is low, risk appetite is probably healthy.
- If VIX rises while drawdown deepens, the setup becomes more defensive.
- If the market is mixed, the regime label avoids false precision and calls it neutral.

## Possible next iterations

- Add sector breadth
- Add option-implied volatility metrics
- Add intraday market data if available
- Convert this notebook into a Streamlit dashboard